# VAE imputation for SUPPORT (PyTorch) — single configuration

This notebook trains **one** VAE on the SUPPORT data exported by `load_support_missing()` and uses it to
**fill in the missing values**. It needs two uploaded files:

* `support_vae.csv` – one row per patient: `split`, `time`, `event`, the 58 encoded features and the 58 `mask_` columns
  (1 = observed, 0 = missing).
* `support_vae_variables.json` – which columns belong to which variable, the variable type
  (`real`, `binary`, `categorical`) and the scaling needed to convert back to original units.

**Architecture** (`METHOD`), as in the missing-data VAE paper:

* `'Zero Imputation'` – the encoder sees the features with missing entries set to 0 (= training mean for real variables).
* `'Encoder Indicators'` – the encoder also gets the missingness mask (one indicator per variable).
* `'Encoder Decoder Indicators'` – as above, and the mask is also fed into the decoder.

**Likelihood**: every variable gets the likelihood that matches its type — Gaussian for real-valued variables,
Bernoulli for binary ones and a softmax/categorical for one-hot groups. Only *observed* variables count in the ELBO,
and each categorical variable counts once.

**Output**: the imputed data set (observed values unchanged, missing values filled by the model), saved both in the
encoded form (same columns as `support_vae.csv`, ready for survival models) and in original units.
Each run also appends its test metrics to `RESULTS_FILE` so methods and seeds can be compared.

In [1]:
import csv
import json
import math
import os
import random
import time
from datetime import datetime

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal, kl_divergence

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

Using device: cuda


## Configuration

In [2]:
# ------------------------------------------------------------------
# Main choices
# ------------------------------------------------------------------
METHOD = 'Encoder Indicators'   # 'Zero Imputation', 'Encoder Indicators' or 'Encoder Decoder Indicators'
SEED = 0                        # controls model init, batching and sampling noise

DATA_FILE = '../data/support_vae.csv'
VARIABLES_FILE = '../data/support_vae_variables.json'

# ------------------------------------------------------------------
# Model and training
# ------------------------------------------------------------------
Z_DIM = 16                      # latent dimension
HIDDEN = (128, 64)              # encoder hidden layers (decoder uses them in reverse)
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
BATCH_SIZE = 128
MAX_EPOCHS = 500
PATIENCE = 1                   # stop after this many epochs without validation-ELBO improvement
VALID_ELBO_SAMPLES = 5          # z samples per patient when computing the validation ELBO (less noisy early stopping)

# ------------------------------------------------------------------
# Evaluation and imputation
# ------------------------------------------------------------------
NUM_IMPORTANCE_SAMPLES = 500    # samples per test patient for the marginal likelihood
IMPUTE_SAMPLES = 500            # z samples per patient used to impute
IMPUTE_IMPORTANCE_WEIGHTED = True  # True: weight samples by how well they explain the observed values (MIWAE-style)
IMPUTE_HARD = True              # True: binaries -> 0/1, categoricals -> one-hot; False: keep probabilities

RESULTS_FILE = 'results_support.csv'   # set to None to not save metrics
OUTPUT_DIR = '.../data'                # where the imputed CSVs are written
SAVE_MODEL_PATH = None          # e.g. 'vae_support.pt'

# ------------------------------------------------------------------
METHOD_OPTIONS = ('Zero Imputation', 'Encoder Indicators', 'Encoder Decoder Indicators')
assert METHOD in METHOD_OPTIONS, METHOD
print(f'SUPPORT | method: {METHOD} | seed: {SEED} | z_dim: {Z_DIM} | hidden: {HIDDEN}')

SUPPORT | method: Encoder Indicators | seed: 0 | z_dim: 16 | hidden: (128, 64)


## Load the data

The features are already encoded (real variables standardized with training statistics, binaries 0/1, categoricals
one-hot) and missing entries are 0. The variable table tells the model which columns belong together.

In [3]:
def set_seed(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)


df = pd.read_csv(DATA_FILE)
with open(VARIABLES_FILE) as f:
  VARIABLES = json.load(f)

FEATURES = [c for c in df.columns if c not in ('split', 'time', 'event') and not c.startswith('mask_')]
MASK_COLS = ['mask_' + c for c in FEATURES]
D = len(FEATURES)
N_VARS = len(VARIABLES)

# --- consistency checks between the CSV and the variable table ---
assert all(c in df.columns for c in MASK_COLS), 'mask columns missing'
assert VARIABLES[-1]['stop'] == D, 'variable table does not match the number of feature columns'
for v in VARIABLES:
  block = df[MASK_COLS[v['start']:v['stop']]].to_numpy()
  assert (block == block[:, :1]).all(), f"mask not constant within variable {v['name']}"

# --- index tensors used by the likelihood ---
REAL_VARS = [v for v in VARIABLES if v['kind'] == 'real']
BIN_VARS = [v for v in VARIABLES if v['kind'] == 'binary']
CAT_VARS = [v for v in VARIABLES if v['kind'] == 'categorical']
VAR_POS = {v['name']: i for i, v in enumerate(VARIABLES)}            # variable -> position in the per-variable mask

REAL_COLS = torch.tensor([v['start'] for v in REAL_VARS], device=DEVICE)
BIN_COLS = torch.tensor([v['start'] for v in BIN_VARS], device=DEVICE)
REAL_VPOS = torch.tensor([VAR_POS[v['name']] for v in REAL_VARS], device=DEVICE)
BIN_VPOS = torch.tensor([VAR_POS[v['name']] for v in BIN_VARS], device=DEVICE)
CAT_SLICES = [(v['start'], v['stop'], VAR_POS[v['name']]) for v in CAT_VARS]
VAR_FIRST_COL = torch.tensor([v['start'] for v in VARIABLES], device=DEVICE)  # column holding each variable's mask


def split_tensors(name):
  part = df[df['split'] == name]
  x = torch.tensor(part[FEATURES].to_numpy(np.float32), device=DEVICE)
  m = torch.tensor(part[MASK_COLS].to_numpy(np.float32), device=DEVICE)
  return x * m, m   # make sure missing entries are exactly 0


X_tr, M_tr = split_tensors('train')
X_va, M_va = split_tensors('valid')
X_te, M_te = split_tensors('test')

print(f'{D} encoded columns, {N_VARS} variables '
      f'({len(REAL_VARS)} real, {len(BIN_VARS)} binary, {len(CAT_VARS)} categorical)')
for name, m in [('train', M_tr), ('valid', M_va), ('test', M_te)]:
  vm = m[:, VAR_FIRST_COL]
  print(f'{name:5s}: {m.shape[0]:5d} patients | observed variables {vm.mean().item():.3f}')

58 encoded columns, 35 variables (26 real, 3 binary, 6 categorical)
train:  6373 patients | observed variables 0.874
valid:   911 patients | observed variables 0.872
test :  1821 patients | observed variables 0.873


In [4]:
# Missingness per variable (training split)
miss = 1 - M_tr[:, VAR_FIRST_COL].mean(0).cpu().numpy()
print(pd.Series(miss, index=[v['name'] for v in VARIABLES])
        .sort_values(ascending=False).round(3).to_string())

adlp        0.618
urine       0.535
glucose     0.495
bun         0.479
alb         0.366
income      0.329
adls        0.314
bili        0.279
pafi        0.253
ph          0.249
edu         0.179
sfdm2       0.156
totcst      0.098
wblc        0.022
charges     0.019
avtisst     0.010
crea        0.007
race        0.004
scoma       0.000
slos        0.000
sex         0.000
age         0.000
diabetes    0.000
hday        0.000
num.co      0.000
dzgroup     0.000
dzclass     0.000
meanbp      0.000
ca          0.000
dementia    0.000
temp        0.000
sod         0.000
hrt         0.000
resp        0.000
adlsc       0.000


## Model

Fully connected encoder and decoder. The decoder outputs one number per encoded column: the mean for real variables,
a logit for binary variables and one logit per category for categorical variables. Each real variable also has a
learned standard deviation.

In [5]:
def mlp(sizes):
  layers = []
  for a, b in zip(sizes[:-1], sizes[1:]):
    layers += [nn.Linear(a, b), nn.ReLU()]
  return nn.Sequential(*layers)


class Encoder(nn.Module):
  def __init__(self, in_dim):
    super().__init__()
    self.body = mlp((in_dim,) + HIDDEN)
    self.mu = nn.Linear(HIDDEN[-1], Z_DIM)
    self.sigma = nn.Linear(HIDDEN[-1], Z_DIM)

  def forward(self, x):
    h = self.body(x)
    return self.mu(h), F.softplus(self.sigma(h)) + 1e-3


class Decoder(nn.Module):
  def __init__(self, b_dim=0):
    super().__init__()
    hidden = HIDDEN[::-1]
    self.first = nn.Sequential(nn.Linear(Z_DIM, hidden[0]), nn.ReLU())
    # the mask (if used) is concatenated after the first layer, like the original
    # notebook concatenated it before the decoder's fine-tune layers
    self.rest = mlp((hidden[0] + b_dim,) + hidden[1:]) if len(hidden) > 1 else nn.Identity()
    self.out = nn.Linear(hidden[-1] if len(hidden) > 1 else hidden[0] + b_dim, D)
    self.log_sigma = nn.Parameter(torch.zeros(len(REAL_VARS)))  # one std per real variable

  def forward(self, z, b=None):
    h = self.first(z)
    if b is not None:
      h = torch.cat([h, b], dim=-1)
    return self.out(self.rest(h))


class VAE(nn.Module):
  def __init__(self, enc_in, dec_b):
    super().__init__()
    self.encoder = Encoder(enc_in)
    self.decoder = Decoder(dec_b)

  def real_sigma(self):
    return self.decoder.log_sigma.clamp(-4.0, 2.0).exp()


def var_mask(m):
  # one indicator per variable, from the column mask
  return m[:, VAR_FIRST_COL]


def get_inputs(x, m, method=None):
  # encoder input and (optional) decoder mask for METHOD
  method = method or METHOD
  x0 = x * m
  if method == 'Zero Imputation':
    return x0, None
  vm = var_mask(m)
  inputs = torch.cat([x0, vm], dim=-1)
  if method == 'Encoder Indicators':
    return inputs, None
  if method == 'Encoder Decoder Indicators':
    return inputs, vm
  raise ValueError(method)


ENC_IN = D + (0 if METHOD == 'Zero Imputation' else N_VARS)
DEC_B = N_VARS if METHOD == 'Encoder Decoder Indicators' else 0

## Likelihood and loss

`log_p_x_given_z` returns, for every patient, the log-likelihood of the **observed** variables only. Real variables use
a Gaussian, binaries a Bernoulli, categoricals a softmax over their one-hot group.

In [6]:
def log_p_x_given_z(model, out, x, m):
  # out: [S, B, D] or [B, D]; x, m: [B, D]. Returns the observed-data log-likelihood.
  vm = m[..., VAR_FIRST_COL]
  total = 0.0

  if len(REAL_VARS):
    mu = out[..., REAL_COLS]
    ll = Normal(mu, model.real_sigma()).log_prob(x[..., REAL_COLS])
    total = total + (ll * vm[..., REAL_VPOS]).sum(-1)

  if len(BIN_VARS):
    logits = out[..., BIN_COLS]
    target = x[..., BIN_COLS].expand_as(logits)   # works for any number of samples
    ll = -F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    total = total + (ll * vm[..., BIN_VPOS]).sum(-1)

  for start, stop, pos in CAT_SLICES:
    ll = (F.log_softmax(out[..., start:stop], dim=-1) * x[..., start:stop]).sum(-1)
    total = total + ll * vm[..., pos]

  return total


def kl_to_prior(q_z):
  p_z = Normal(torch.zeros_like(q_z.loc), torch.ones_like(q_z.scale))
  return kl_divergence(q_z, p_z).sum(-1)


def elbo_terms(model, x, m, n_samples=1):
  # Returns per-patient (log p(x_o|z) averaged over samples, KL).
  inputs, dec_b = get_inputs(x, m)
  mu, sigma = model.encoder(inputs)
  q_z = Normal(mu, sigma)
  z = q_z.rsample((n_samples,))                          # [S, B, Z]
  b = None if dec_b is None else dec_b.expand(n_samples, *dec_b.shape)
  out = model.decoder(z, b)                              # [S, B, D]
  log_px = log_p_x_given_z(model, out, x, m).mean(0)     # [B]
  return log_px, kl_to_prior(q_z)

## Training and evaluation functions

In [7]:
def batches(n, batch_size, shuffle):
  order = torch.randperm(n, device=DEVICE) if shuffle else torch.arange(n, device=DEVICE)
  for start in range(0, n, batch_size):
    yield order[start:start + batch_size]


def train_epoch(model, optimizer):
  model.train()
  total, count = 0.0, 0
  for idx in batches(X_tr.shape[0], BATCH_SIZE, shuffle=True):
    log_px, kl = elbo_terms(model, X_tr[idx], M_tr[idx])
    loss = (kl - log_px).mean()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 2.5)
    optimizer.step()
    total += -loss.item() * len(idx)
    count += len(idx)
  return total / count   # training ELBO per patient


@torch.inference_mode()
def evaluate(model, X, M, n_samples=1):
  # Mean ELBO, log p(x_o|z) and KL per patient.
  model.eval()
  log_px, kl = elbo_terms(model, X, M, n_samples)
  return {'elbo': (log_px - kl).mean().item(), 'log_prob': log_px.mean().item(),
          'kl': kl.mean().item()}


@torch.inference_mode()
def posterior_samples(model, x, m, s):
  # Draws s samples z ~ q(z|x_o) and returns decoder outputs and log importance weights.
  inputs, dec_b = get_inputs(x, m)
  mu, sigma = model.encoder(inputs)
  q_z = Normal(mu, sigma)
  z = q_z.sample((s,))                                   # [S, B, Z]
  b = None if dec_b is None else dec_b.expand(s, *dec_b.shape)
  out = model.decoder(z, b)                              # [S, B, D]
  log_px = log_p_x_given_z(model, out, x, m)             # [S, B]
  log_pz = Normal(torch.zeros_like(z), torch.ones_like(z)).log_prob(z).sum(-1)
  log_qz = q_z.log_prob(z).sum(-1)
  return out, log_px + log_pz - log_qz


@torch.inference_mode()
def evaluate_test(model, X, M, s=NUM_IMPORTANCE_SAMPLES, chunk=256):
  # Importance-sampled marginal likelihood of the observed values, per patient.
  model.eval()
  mll, n_obs = [], []
  for idx in batches(X.shape[0], chunk, shuffle=False):
    _, log_w = posterior_samples(model, X[idx], M[idx], s)
    mll.append(torch.logsumexp(log_w, dim=0) - math.log(s))
    n_obs.append(var_mask(M[idx]).sum(-1))
  mll, n_obs = torch.cat(mll), torch.cat(n_obs)
  res = evaluate(model, X, M, n_samples=10)
  res['marginal_ll'] = mll.mean().item()
  res['nats_per_observed_var'] = (-mll / n_obs).mean().item()
  return res


@torch.inference_mode()
def reconstruction_check(model, X, M):
  # How well the model reproduces the *observed* values (a sanity check, not imputation quality).
  model.eval()
  inputs, dec_b = get_inputs(X, M)
  out = model.decoder(model.encoder(inputs)[0], dec_b)   # decode the posterior mean
  vm = var_mask(M)
  res = {}
  if len(REAL_VARS):
    err = (out[:, REAL_COLS] - X[:, REAL_COLS]) ** 2
    w = vm[:, REAL_VPOS]
    res['rmse_real_observed'] = (err * w).sum().div(w.sum()).sqrt().item()  # in standard deviations
  correct, total = 0.0, 0.0
  for start, stop, pos in CAT_SLICES:
    hit = (out[:, start:stop].argmax(-1) == X[:, start:stop].argmax(-1)).float()
    correct += (hit * vm[:, pos]).sum().item()
    total += vm[:, pos].sum().item()
  if len(BIN_VARS):
    hit = ((out[:, BIN_COLS] > 0).float() == X[:, BIN_COLS]).float()
    correct += (hit * vm[:, BIN_VPOS]).sum().item()
    total += vm[:, BIN_VPOS].sum().item()
  res['acc_discrete_observed'] = correct / max(total, 1)
  return res

## Train

In [8]:
set_seed(SEED)
model = VAE(ENC_IN, DEC_B).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')

best_valid_elbo, best_state, best_epoch = None, None, None
num_epochs_since_improvement = 0
t_start = time.time()

for epoch in range(MAX_EPOCHS):
  train_elbo = train_epoch(model, optimizer)
  valid_elbo = evaluate(model, X_va, M_va, n_samples=VALID_ELBO_SAMPLES)['elbo']

  improved = best_valid_elbo is None or valid_elbo > best_valid_elbo
  if improved or epoch % 10 == 0:
    print(f'epoch {epoch:3d} | train ELBO {train_elbo:9.3f} | valid ELBO {valid_elbo:9.3f}'
          f'{" *" if improved else ""}')

  if improved:
    best_valid_elbo, best_epoch = valid_elbo, epoch
    num_epochs_since_improvement = 0
    best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
  elif num_epochs_since_improvement >= PATIENCE:
    print('Early stopping.')
    break
  else:
    num_epochs_since_improvement += 1

epochs_trained = epoch + 1
train_time = time.time() - t_start
model.load_state_dict(best_state)
print(f'Trained {epochs_trained} epochs in {train_time:.0f}s; '
      f'best epoch {best_epoch} (valid ELBO {best_valid_elbo:.3f})')

if SAVE_MODEL_PATH:
  torch.save({'state_dict': best_state, 'METHOD': METHOD, 'SEED': SEED,
              'Z_DIM': Z_DIM, 'HIDDEN': HIDDEN}, SAVE_MODEL_PATH)
  print('Saved model to', SAVE_MODEL_PATH)

39,284 parameters
epoch   0 | train ELBO   -40.524 | valid ELBO   -39.152 *
epoch   1 | train ELBO   -38.999 | valid ELBO   -38.424 *
epoch   2 | train ELBO   -38.310 | valid ELBO   -38.036 *
epoch   3 | train ELBO   -37.999 | valid ELBO   -37.771 *
epoch   4 | train ELBO   -37.764 | valid ELBO   -37.451 *
epoch   5 | train ELBO   -37.429 | valid ELBO   -37.226 *
epoch   6 | train ELBO   -37.230 | valid ELBO   -37.022 *
epoch   7 | train ELBO   -37.111 | valid ELBO   -36.962 *
epoch   8 | train ELBO   -37.020 | valid ELBO   -36.744 *
epoch   9 | train ELBO   -36.820 | valid ELBO   -36.602 *
epoch  10 | train ELBO   -36.645 | valid ELBO   -36.452 *
epoch  11 | train ELBO   -36.434 | valid ELBO   -36.333 *
epoch  12 | train ELBO   -36.349 | valid ELBO   -36.159 *
epoch  13 | train ELBO   -36.167 | valid ELBO   -36.058 *
epoch  14 | train ELBO   -36.049 | valid ELBO   -35.916 *
epoch  15 | train ELBO   -35.909 | valid ELBO   -35.747 *
epoch  16 | train ELBO   -35.740 | valid ELBO   -35.65

## Test evaluation

* `marginal_ll` – importance-sampled log-likelihood of each test patient's observed values (higher is better).
  `nats_per_observed_var` divides it by the number of observed variables, so patients with many gaps are comparable.
* `rmse_real_observed` / `acc_discrete_observed` – how well the model *reconstructs observed* values. This is only a
  sanity check: the quality of the imputations themselves is judged later, through the survival models.

In [9]:
t0 = time.time()
res = evaluate_test(model, X_te, M_te)
res.update(reconstruction_check(model, X_te, M_te))
print(f'Test evaluation: {time.time() - t0:.0f}s\n')
print(f'SUPPORT | {METHOD} | seed {SEED}')
for k, v in res.items():
  print(f'  {k:24s} {v:.4f}')

Test evaluation: 0s

SUPPORT | Encoder Indicators | seed 0
  elbo                     -32.1301
  log_prob                 -26.7294
  kl                       5.4007
  marginal_ll              -31.3492
  nats_per_observed_var    1.0234
  rmse_real_observed       0.7491
  acc_discrete_observed    0.7643


## Impute

For every patient, `IMPUTE_SAMPLES` latent samples are drawn from the encoder and decoded into expected values
(real: mean; binary: probability; categorical: class probabilities). With `IMPUTE_IMPORTANCE_WEIGHTED = True` the
samples are weighted by how well they explain that patient's observed values (self-normalised importance weights, as
in MIWAE); otherwise they are averaged equally. Observed values are always kept exactly as they were.

In [10]:
def expected_values(out):
  # Decoder outputs -> expected value of every encoded column.
  ev = torch.empty_like(out)
  if len(REAL_VARS):
    ev[..., REAL_COLS] = out[..., REAL_COLS]
  if len(BIN_VARS):
    ev[..., BIN_COLS] = torch.sigmoid(out[..., BIN_COLS])
  for start, stop, _ in CAT_SLICES:
    ev[..., start:stop] = torch.softmax(out[..., start:stop], dim=-1)
  return ev


def harden(pred):
  pred = pred.clone()
  if len(BIN_VARS):
    pred[:, BIN_COLS] = (pred[:, BIN_COLS] >= 0.5).float()
  for start, stop, _ in CAT_SLICES:
    pred[:, start:stop] = F.one_hot(pred[:, start:stop].argmax(-1), stop - start).float()
  return pred


@torch.inference_mode()
def impute(model, X, M, s=IMPUTE_SAMPLES, chunk=256):
  model.eval()
  filled = []
  for idx in batches(X.shape[0], chunk, shuffle=False):
    x, m = X[idx], M[idx]
    out, log_w = posterior_samples(model, x, m, s)       # [S, B, D], [S, B]
    if IMPUTE_IMPORTANCE_WEIGHTED:
      w = torch.softmax(log_w, dim=0)
    else:
      w = torch.full_like(log_w, 1.0 / s)
    pred = (w[..., None] * expected_values(out)).sum(0)  # [B, D]
    if IMPUTE_HARD:
      pred = harden(pred)
    filled.append(m * x + (1 - m) * pred)
  return torch.cat(filled)


set_seed(SEED)
t0 = time.time()
imputed = {name: impute(model, X, M) for name, X, M in
           [('train', X_tr, M_tr), ('valid', X_va, M_va), ('test', X_te, M_te)]}
print(f'Imputation: {time.time() - t0:.0f}s')

# observed values must be untouched
for (name, X, M) in [('train', X_tr, M_tr), ('valid', X_va, M_va), ('test', X_te, M_te)]:
  assert torch.equal(imputed[name] * M, X * M), name
  assert not torch.isnan(imputed[name]).any(), name
print('Observed values unchanged, no NaNs.')

Imputation: 0s
Observed values unchanged, no NaNs.


## Convert back to original units and save

Two files per run:

* `support_imputed_<method>_seed<seed>.csv` – same layout as `support_vae.csv` (`split`, `time`, `event`, the 58
  encoded features now without gaps, and the `mask_` columns so you still know what was imputed). Use this for the
  survival models.
* `support_imputed_<method>_seed<seed>_original_units.csv` – one readable column per variable (lab values in their
  units, `male`/`female`, disease names, …), plus `split`, `time`, `event`.

In [11]:
def to_original_units(X):
  # Encoded matrix (numpy, [n, D]) -> DataFrame with one column per original variable.
  out = {}
  for v in VARIABLES:
    block = X[:, v['start']:v['stop']]
    if v['kind'] == 'real':
      x = block[:, 0] * v['std'] + v['mean']
      out[v['name']] = np.expm1(x) if v['log'] else x
    elif v['kind'] == 'binary':
      pos = v['positive']
      neg = {'male': 'female', 1: 0}.get(pos, f'not {pos}')
      out[v['name']] = np.where(block[:, 0] >= 0.5, pos, neg)
    else:
      out[v['name']] = np.asarray(v['categories'], dtype=object)[block.argmax(1)]
  return pd.DataFrame(out)


tag = f"{METHOD.lower().replace(' ', '_')}_seed{SEED}"
encoded_parts, original_parts = [], []
for name in ['train', 'valid', 'test']:
  part = df[df['split'] == name].reset_index(drop=True)
  Ximp = imputed[name].cpu().numpy()
  meta = part[['split', 'time', 'event']]
  encoded_parts.append(pd.concat([meta, pd.DataFrame(Ximp, columns=FEATURES), part[MASK_COLS]], axis=1))
  original_parts.append(pd.concat([meta, to_original_units(Ximp)], axis=1))

os.makedirs(OUTPUT_DIR, exist_ok=True)
encoded_path = os.path.join(OUTPUT_DIR, f'support_imputed_{tag}.csv')
original_path = os.path.join(OUTPUT_DIR, f'support_imputed_{tag}_original_units.csv')
pd.concat(encoded_parts, ignore_index=True).to_csv(encoded_path, index=False)
pd.concat(original_parts, ignore_index=True).to_csv(original_path, index=False)
print('Saved', encoded_path)
print('Saved', original_path)

Saved .../data/support_imputed_encoder_indicators_seed0.csv
Saved .../data/support_imputed_encoder_indicators_seed0_original_units.csv


In [12]:
# Look at a few test patients: imputed values are marked with *
part = df[df['split'] == 'test'].reset_index(drop=True)
orig = to_original_units(imputed['test'].cpu().numpy())
vmask = var_mask(M_te).cpu().numpy().astype(bool)
show = [v['name'] for v in VARIABLES][:14]
rows = []
for i in range(5):
  row = {}
  for j, v in enumerate(VARIABLES):
    if v['name'] in show:
      val = orig.loc[i, v['name']]
      val = f'{val:.2f}' if isinstance(val, (float, np.floating)) else str(val)
      row[v['name']] = val if vmask[i, j] else val + ' *'
  rows.append(row)
print(pd.DataFrame(rows).to_string())

     age     sex    slos            dzgroup             dzclass num.co      edu        income   scoma    charges     totcst avtisst   race   hday
0  78.79  female    7.00  ARF/MOSF w/Sepsis            ARF/MOSF   2.00  11.79 *  under $11k *   41.00   77543.03   46027.35   14.50  black  15.00
1  32.97    male   20.00               Coma                Coma   0.00    14.00      $11-$25k  100.00   74168.52   50308.76   36.67  white   4.00
2  80.46  female   18.00  ARF/MOSF w/Sepsis            ARF/MOSF   3.00     8.00    under $11k    0.00   30254.99   16452.51   24.67  white   1.00
3  39.64    male   48.00          Cirrhosis  COPD/CHF/Cirrhosis   2.00    13.00    $11-$25k *    0.00  272602.03  133509.59   13.75  white   1.00
4  48.00    male  117.00                CHF  COPD/CHF/Cirrhosis   1.00  12.04 *  under $11k *    0.00  337299.41  222272.66   33.50  white   1.00


## Save metrics

In [13]:
METRIC_NAMES = ['elbo', 'log_prob', 'kl', 'marginal_ll', 'nats_per_observed_var',
                'rmse_real_observed', 'acc_discrete_observed']

if RESULTS_FILE:
  row = {'timestamp': datetime.now().isoformat(timespec='seconds'),
         'method': METHOD, 'seed': SEED, 'z_dim': Z_DIM, 'hidden': '-'.join(map(str, HIDDEN)),
         'max_epochs': MAX_EPOCHS, 'patience': PATIENCE,
         'epochs_trained': epochs_trained, 'best_epoch': best_epoch,
         'train_time_s': round(train_time, 1), 'imputed_file': encoded_path}
  row.update({name: res.get(name, float('nan')) for name in METRIC_NAMES})
  new_file = not os.path.exists(RESULTS_FILE)
  with open(RESULTS_FILE, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(row))
    if new_file:
      writer.writeheader()
    writer.writerow(row)
  print('Appended results to', RESULTS_FILE)

Appended results to results_support.csv


end of notebook